# synthCMRO2: figures and statistics from the public data

This notebook reproduces Figures 2–4 and S2 and Table S2 of *Generative Deep Learning Maps Brain Oxygen
Metabolism and Its Age-Related Decline from Two Standard Noninvasive MRI Scans*
([medRxiv preprint](https://doi.org/10.64898/2026.11.17.26368289)) from the shared group tables in
`data/public/`. It runs in the browser (JupyterLite) or on Binder; nothing is installed on your computer.

Run the cells from top to bottom (**Run → Run All Cells**). In the browser, the first cell downloads the
Python packages (about 30–60 s); each figure then takes from a few seconds to a few minutes, depending on the
computer. Each figure cell shows the figure produced here; the published figures are in `reference/figures/`.

In [ ]:
import os, sys, time
from pathlib import Path

if "pyodide" in sys.modules:  # JupyterLite: install the packages in the browser
    import piplite
    # versions compatible with the packages built into the browser Python (Pyodide)
    await piplite.install(["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pyyaml",
                           "seaborn==0.13.2", "nibabel==5.3.2", "nilearn==0.11.1"])
    await piplite.install("synthcmro2", deps=False)  # wheel built from this repository

    # Work on an in-memory copy of the files. The browser file storage delivers binary files (.gz, .nii, .npz)
    # as text, which corrupts them, so these are downloaded from the repository instead.
    import shutil
    from pyodide.http import pyfetch

    REPO = "https://raw.githubusercontent.com/NeuroenergeticsLab/synthCMRO2/main/"
    drive, work = Path.cwd(), Path("/home/pyodide/synthCMRO2")
    for src in sorted(drive.rglob("*")):
        rel = src.relative_to(drive)
        if src.is_dir() or rel.parts[0] == "results" or rel.suffix in (".ipynb", ".png"):
            continue
        dst = work / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.suffix in (".gz", ".nii", ".npz"):
            response = await pyfetch(REPO + rel.as_posix())
            if not response.ok:
                raise OSError(f"download failed ({response.status}): {REPO + rel.as_posix()}")
            dst.write_bytes(await response.bytes())
        else:
            shutil.copyfile(src, dst)
    os.chdir(work)

# project root: the folder that contains config/ and data/
root = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
os.chdir(root)
print("project root:", root)

## Settings

`FAST = True` (default) uses 200 instead of 2000 bootstrap resamples for the 95% CIs of the group-level
correlations: quicker, but these CIs then differ slightly from the published ones. Set `FAST = False` to
reproduce every number of the article exactly.

In [ ]:
FAST = True
DPI = 150  # resolution of the PNG figures shown here (the article uses 300)

import logging
import yaml
from IPython.display import Image, display
from synthcmro2.figures.commands import benchmark_main, figures_main

cfg = yaml.safe_load((root / "config/figures.yaml").read_text(encoding="utf8"))
cfg["paths"]["output_dir"] = "results/browser"
cfg["plotting"]["formats"] = ["png"]
if FAST:
    cfg["analysis"]["n_bootstrap"] = 200
config = root / "config/figures_browser.yaml"
config.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf8")
out = root / "results/browser"


def figure(number):
    """Build one figure with its statistics tables and show it."""
    start = time.time()
    figures_main(["--config", str(config), "--figures", number, "--dpi", str(DPI)])
    logging.getLogger("synthcmro2").setLevel(logging.WARNING)  # quieter after the first run
    display(Image(filename=str(out / "figures" / f"Figure{number}.png"), width=900))
    print(f"Figure {number}: {time.time() - start:.0f} s; statistics in {out / 'stats'}")

## Figure 2: agreement in the test set

Group-level and participant-wise agreement between synthetic and analytical CMRO₂ for every input
configuration, the cortical maps, and the comparison with the single-input models (T1-weighted, fMRI).

In [ ]:
figure("2")

## Figure 3: calculation task

Agreement during the calculation task and the PLS of calculation versus rest (precomputed PLS results in
`data/public/pls/`).

In [ ]:
figure("3")

## Figure 4: generalization and age-related decline

Same-protocol and cross-vendor datasets, and the age-related decline of analytical and synthetic CMRO₂.

In [ ]:
figure("4")

## Figure S2: test-retest reliability

Parcel-wise ICC(2,1) of analytical and synthetic CMRO₂ between two within-session scans.

In [ ]:
figure("S2")

## Table S2: model errors versus the test-retest error

In [ ]:
import pandas as pd

benchmark_main(["--config", str(config)])
pd.read_csv(out / "stats/noise_benchmark.csv")

## Check against the published statistics

Compares every statistics table produced here with `reference/stats/` (relative tolerance 1e-6) and lists the
columns that differ. With `FAST = True`, only bootstrap confidence intervals differ; with `FAST = False`,
every table matches.

In [ ]:
import numpy as np

rows = []
for ref_file in sorted((root / "reference/stats").glob("*.csv")):
    new_file = out / "stats" / ref_file.name
    if not new_file.exists():
        continue
    ref, new = pd.read_csv(ref_file), pd.read_csv(new_file)
    if ref.shape != new.shape or list(ref.columns) != list(new.columns):
        rows.append({"table": ref_file.name, "matches the article": False, "differing columns": "table layout"})
        continue
    differ = [
        c for c in ref.columns
        if not (np.allclose(ref[c], new[c], rtol=1e-6, equal_nan=True) if pd.api.types.is_numeric_dtype(ref[c])
                else ref[c].astype(str).equals(new[c].astype(str)))
    ]
    rows.append({"table": ref_file.name, "matches the article": not differ, "differing columns": ", ".join(differ)})
pd.DataFrame(rows)